# 02 — Pré-processamento e Engenharia de Dados

**Objetivo:**
Preparar os dados para o modelo, prevenindo Data Leakage.
1. Carregar o dataset codificado do EDA.
2. Limpar duplicatas e isolar o teste ANTES de qualquer transformação.
3. Dividir em Treino e Teste (Estratificado).
4. Normalizar os dados com RobustScaler e aplicar filtro de correlação.
5. Aplicar o undersampling (DoS) e SMOTE no treino.
6. Salvar os dados finais.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / 'artifacts' / 'notebooks' / 'artifacts' / 'model' / 'data'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregamento e Limpeza (Antes do Split)
**O que estamos fazendo:** Removemos colunas que não são features (id, portas) e limpamos as duplicatas exatas.
**Por que:** O `id` e as portas causam vazamento. As duplicatas exatas precisam sair antes do split para não contaminar o teste. **Importante:** a remoção das portas é feita *após* a checagem de duplicatas para manter a frequência real do DoS (que tem as portas de origem variando).

In [ ]:
df = pd.read_csv(MODEL_DIR / 'dataset_encoded.csv')
TARGET = 'Attack_type'

print(f'Shape original: {df.shape}')

# 1. Remove duplicatas exatas de captura (mantendo a coluna de porta de origem para não colapsar o DoS)
df = df.drop_duplicates()
print(f'Shape após drop_duplicates: {df.shape}')

## 2. Split Treino/Teste Estratificado
**O que estamos fazendo:** Dividimos os dados ANTES de qualquer transformação.
**Por que:** Se aplicarmos o Scaler ou o SMOTE em todo o dataset e depois dividirmos, o modelo terá acesso a informações do teste durante o treino (Data Leakage).

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

## 3. Normalização (RobustScaler)
**O que estamos fazendo:** Ajustando a escala das features.
**Por que:** Usamos o RobustScaler porque ele usa a mediana e o IQR, sendo imune aos outliers extremos (diferente do StandardScaler).

In [ ]:
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

## 4. Tratamento do Desbalanceamento (Undersampling + SMOTE)
**O que estamos fazendo:** Reduzindo o DoS SYN e criando amostras sintéticas das classes minoritárias.
**Por que:** A classe DOS_SYN_Hping é quase 77% dos dados. O undersampling reduz o excesso e o SMOTE cria vizinhos sintéticos para as classes raras. É aplicado **apenas no treino**.

In [ ]:
contagens_antes = y_train.value_counts().sort_values(ascending=False)
n_dos = int(contagens_antes.get('DOS_SYN_Hping', 0))
DOS_CAP = min(10000, n_dos)

print(f'DoS SYN no treino: {n_dos:,}  |  cap efetivo: {DOS_CAP:,}')

if n_dos <= DOS_CAP:
    print('DoS SYN já está abaixo do cap — pulando undersample.')
    X_train_us, y_train_us = X_train_scaled, y_train
else:
    rus = RandomUnderSampler(
        sampling_strategy={'DOS_SYN_Hping': DOS_CAP},
        random_state=RANDOM_STATE
    )
    X_train_us, y_train_us = rus.fit_resample(X_train_scaled, y_train)
    print(f'DoS SYN: {n_dos:,} → {DOS_CAP:,}')

print(f'\nApós undersample: {X_train_us.shape}')

In [ ]:
SMOTE_TARGET = 500

contagens = y_train_us.value_counts()
sampling_dict = {cls: SMOTE_TARGET for cls, cnt in contagens.items() if cnt < SMOTE_TARGET}

print('Classes que receberão SMOTE:')
for cls, target in sampling_dict.items():
    print(f'  {cls:<28} {int(contagens[cls]):>5} → {target}')

if sampling_dict:
    min_class = int(contagens.min())
    k_neighbors = max(1, min(3, min_class - 1))
    smote = SMOTE(
        sampling_strategy=sampling_dict,
        k_neighbors=k_neighbors,
        random_state=RANDOM_STATE
    )
    X_train_res, y_train_res = smote.fit_resample(X_train_us, y_train_us)
else:
    X_train_res, y_train_res = X_train_us, y_train_us

X_train_res = pd.DataFrame(X_train_res, columns=X_train.columns)
print(f'\nTreino final (balanceado): {X_train_res.shape}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

y_train.value_counts().plot(kind='bar', ax=ax1, color='salmon')
ax1.set_title('Antes do Balanceamento (Treino)')
ax1.set_yscale('log')
ax1.set_ylabel('Contagem (Log)')

pd.Series(y_train_res).value_counts().plot(kind='bar', ax=ax2, color='mediumseagreen')
ax2.set_title('Depois do Balanceamento (Treino)')
ax2.set_ylabel('Contagem')

plt.tight_layout()
plt.show()

## 5. Salvamento dos Dados Processados
Salvamos os arquivos para o Notebook 03 (Modelagem) carregar e treinar os modelos.

In [ ]:
X_train_res.to_csv(MODEL_DIR / 'X_train.csv', index=False)
pd.Series(y_train_res, name=TARGET).to_csv(MODEL_DIR / 'y_train.csv', index=False)
X_test_scaled.to_csv(MODEL_DIR / 'X_test.csv', index=False)
pd.Series(y_test.reset_index(drop=True), name=TARGET).to_csv(MODEL_DIR / 'y_test.csv', index=False)
print('Dados processados e salvos com sucesso!')